# DualGNN inference demo

Load the weights and sample 8 fine regular triangulations of `[0,4]^2`.

The shipped checkpoints were trained on polygons with `Npts` in `[5, 40]`;
the model extrapolates beyond that but quality is not guaranteed.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

from dualgnn import DualGraph, geometry, sample
from dualgnn.model import DualGNN

## Config

In [ ]:
# CKPT is relative to this notebook (which lives in `notebooks/`).
# Run with the kernel started here, OR set CKPT to an absolute path.
CKPT = Path("../ckpts/reinforce.pt")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

Ntriangs = 8
verts    = [[0,0],[0,4],[4,0],[4,4]]  # vertices of [0,4]^2


## Construct the DualGraph

`enum_lattice_pts(verts)` takes the convex hull of `verts` and returns
every lattice point inside it. Non-convex inputs are silently replaced
by their hull.


In [ ]:
pts   = geometry.enum_lattice_pts(verts)
cmplx = DualGraph(pts)

print(f"polygon  Npts={len(pts)}  candidate simps={cmplx.simps.shape[0]}  "
      f"simps per FRT={cmplx.N_simps_per_ft}")


## Load the model

In [ ]:
net = DualGNN.from_ckpt(CKPT, DEVICE)
print(f"model  D={net.D}  K={net.K}  "
      f"params={sum(p.numel() for p in net.parameters()):,}  device={DEVICE}")

## Sample and plot FRTs

Sample

In [ ]:
fts = sample(net, cmplx, Ntriangs=Ntriangs)
print(f"sampled: {fts.shape}  dtype={fts.dtype}")
print(f"unique:  {len({fts[i].tobytes() for i in range(Ntriangs)})}/{Ntriangs}")

Plot

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, ft in zip(axes.flat, fts):
    for simp in ft:
        tri = pts[simp]
        ax.fill(tri[:, 0], tri[:, 1], edgecolor="k", fill=False)
    ax.scatter(pts[:, 0], pts[:, 1], s=8, color="k")
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()
